# BKT training, evaluation, and model selection

Trains BKT models on the cleaned algebra datasets, compares six candidates (three datasets × two model variants), picks the best one, and saves the final parameters.

The six candidates are the 2006-07, 2008-09, and merged 2006-09 datasets, each fitted with a basic BKT model and a version that includes a forgetting parameter.

## 1. Imports, configuration, and paths

In [ ]:

from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyBKT.models import Model

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True


In [ ]:

# -----------------------------
# Project paths
# -----------------------------
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("MODELS_DIR:", MODELS_DIR)

# -----------------------------
# Main input files
# -----------------------------
TRAIN_06_FILENAME = "df_train_kc_cleaned_2006-2007_v.1.csv"
TRAIN_08_FILENAME = "df_train_kc_cleaned_2008-2009_v.1.csv"

# -----------------------------
# Reproducibility and fitting options
# -----------------------------
RANDOM_STATE = 42
VALID_SIZE = 0.20
NUM_FITS = 5
REQUIRE_EQUALS_IN_STEP = True


## 2. Load the cleaned datasets

In [ ]:

train_06_path = PROCESSED_DIR / TRAIN_06_FILENAME
train_08_path = PROCESSED_DIR / TRAIN_08_FILENAME

df_06_raw = pd.read_csv(train_06_path, encoding="latin")
df_08_raw = pd.read_csv(train_08_path, encoding="latin")

datasets_raw = {
    "2006-07": df_06_raw,
    "2008-09": df_08_raw,
    "2006-09": pd.concat([df_06_raw, df_08_raw], ignore_index=True),
}

for name, df_ in datasets_raw.items():
    print(f"{name}: shape={df_.shape}")

display(df_06_raw.head())


In [ ]:

required_columns = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "Correct First Attempt",
    "Step Start Time",
    "skill_name",
]

for dataset_name, df_ in datasets_raw.items():
    missing_required = [c for c in required_columns if c not in df_.columns]
    if missing_required:
        raise ValueError(f"{dataset_name} is missing required columns: {missing_required}")

print("All required columns found in every dataset.")


## 3. Helper functions

In [ ]:

KC_NAME_MAP = {
    "Expand / Eliminate Parentheses": "expand_eliminate_parentheses",
    "Combine Like Terms": "combine_like_terms",
    "Move Constants Across the Equation": "move_constants",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

VALID_SKILLS = list(KC_NAME_MAP.keys())
VALID_SKILLS_STD = list(KC_NAME_MAP.values())

DEFAULTS = {
    "order_id": "order_id",
    "user_id": "user_id",
    "skill_name": "skill_name_std",
    "correct": "correct",
}

MODEL_SPECS = {
    "basic": {},
    "forgets": {"forgets": True},
}

MODEL_COMPLEXITY = {
    "basic": 0,
    "forgets": 1,
}


In [ ]:

def prepare_columns(df):
    df = df.copy()

    time_cols = [
        "Step Start Time",
        "First Transaction Time",
        "Correct Transaction Time",
        "Step End Time",
    ]
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def standardize_skill_name(skill_name):
    if pd.isna(skill_name):
        return np.nan
    skill_name = str(skill_name).strip()
    if skill_name in KC_NAME_MAP:
        return KC_NAME_MAP[skill_name]
    if skill_name in KC_NAME_MAP.values():
        return skill_name
    return np.nan


def build_pybkt_df(
    df,
    skill_subset=None,
    require_equals_in_step=True,
    min_student_skill_interactions=1,
):
    data = df.copy()

    data = data.dropna(subset=[
        "Anon Student Id",
        "Problem Name",
        "Step Name",
        "Correct First Attempt",
        "Step Start Time",
        "skill_name",
    ]).copy()

    if require_equals_in_step:
        data = data[data["Step Name"].astype(str).str.contains("=", na=False)].copy()

    data["Correct First Attempt"] = pd.to_numeric(data["Correct First Attempt"], errors="coerce")
    data = data[data["Correct First Attempt"].isin([0, 1])].copy()

    data["skill_name_std"] = data["skill_name"].apply(standardize_skill_name)
    data = data[data["skill_name_std"].notna()].copy()

    if skill_subset is not None:
        skill_subset_std = [standardize_skill_name(s) for s in skill_subset]
        skill_subset_std = [s for s in skill_subset_std if pd.notna(s)]
        data = data[data["skill_name_std"].isin(skill_subset_std)].copy()

    sort_cols = [
        c for c in [
            "Anon Student Id",
            "Problem Name",
            "Step Start Time",
            "First Transaction Time",
            "Correct Transaction Time",
            "Step End Time",
            "Row",
        ]
        if c in data.columns
    ]
    data = data.sort_values(sort_cols).reset_index(drop=True)

    data["user_id"] = data["Anon Student Id"].astype(str)
    data["correct"] = data["Correct First Attempt"].astype(int)
    data["order_id"] = np.arange(1, len(data) + 1)

    if min_student_skill_interactions > 1:
        seq_lengths = (
            data.groupby(["user_id", "skill_name_std"])
            .size()
            .reset_index(name="n_interactions")
        )
        keep_pairs = seq_lengths[seq_lengths["n_interactions"] >= min_student_skill_interactions][
            ["user_id", "skill_name_std"]
        ]
        data = data.merge(keep_pairs, on=["user_id", "skill_name_std"], how="inner").copy()
        data = data.sort_values(sort_cols).reset_index(drop=True)
        data["order_id"] = np.arange(1, len(data) + 1)

    bkt_df = data[["order_id", "user_id", "skill_name_std", "correct"]].copy()
    return data, bkt_df


def train_valid_split_by_student(df, valid_size=0.20, seed=42):
    users = pd.Series(df["Anon Student Id"].dropna().astype(str).unique())
    shuffled_users = users.sample(frac=1, random_state=seed).tolist()
    n_valid = max(1, int(len(shuffled_users) * valid_size))

    valid_users = set(shuffled_users[:n_valid])
    train_users = set(shuffled_users[n_valid:])

    train_df = df[df["Anon Student Id"].astype(str).isin(train_users)].copy()
    valid_df = df[df["Anon Student Id"].astype(str).isin(valid_users)].copy()
    return train_df, valid_df


def summarize_interactions_per_student_skill(prepared_df):
    return (
        prepared_df.groupby(["skill_name_std", "user_id"])
        .size()
        .reset_index(name="n_interactions")
    )


def extract_param_table(model):
    params = model.params().reset_index().copy()
    if "skill" in params.columns:
        params = params.rename(columns={"skill": "skill_name_std"})
    if "param" in params.columns:
        params = params.rename(columns={"param": "parameter"})
    if "value" not in params.columns:
        params = params.rename(columns={params.columns[-1]: "value"})
    return params


def safe_evaluate(model, data, metric):
    try:
        return float(model.evaluate(data=data, defaults=DEFAULTS, metric=metric))
    except Exception as exc:
        print(f"Could not compute metric='{metric}': {exc}")
        return np.nan


def safe_predict(model, data):
    try:
        return model.predict(data=data, defaults=DEFAULTS)
    except Exception as exc:
        print(f"Prediction skipped: {exc}")
        return None


def parameter_diagnostics(param_df):
    pivot = (
        param_df.pivot_table(
            index="skill_name_std",
            columns="parameter",
            values="value",
            aggfunc="first",
        )
        .reset_index()
    )

    for col in ["prior", "learns", "guesses", "slips", "forgets"]:
        if col not in pivot.columns:
            pivot[col] = np.nan

    diagnostics = {
        "mean_guess": pivot["guesses"].mean(),
        "max_guess": pivot["guesses"].max(),
        "mean_slip": pivot["slips"].mean(),
        "max_slip": pivot["slips"].max(),
        "mean_learn": pivot["learns"].mean(),
        "max_learn": pivot["learns"].max(),
        "n_skills": pivot["skill_name_std"].nunique(),
    }
    return diagnostics, pivot


def fit_one_dataset_model(
    dataset_name,
    df_raw,
    model_name,
    model_fit_kwargs,
    valid_size=0.20,
    seed=42,
    num_fits=5,
):
    df_clean = prepare_columns(df_raw)
    train_df, valid_df = train_valid_split_by_student(df_clean, valid_size=valid_size, seed=seed)

    train_prepared, train_bkt = build_pybkt_df(
        train_df,
        require_equals_in_step=REQUIRE_EQUALS_IN_STEP,
        min_student_skill_interactions=1,
    )
    valid_prepared, valid_bkt = build_pybkt_df(
        valid_df,
        require_equals_in_step=REQUIRE_EQUALS_IN_STEP,
        min_student_skill_interactions=1,
    )

    if len(train_bkt) == 0 or len(valid_bkt) == 0:
        raise ValueError(f"{dataset_name} - {model_name}: empty train or validation set after filtering.")

    model = Model(seed=seed, num_fits=num_fits)
    model.fit(data=train_bkt, defaults=DEFAULTS, **model_fit_kwargs)

    param_df = extract_param_table(model)
    diag, param_pivot = parameter_diagnostics(param_df)

    summary = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "candidate_name": f"{dataset_name}__{model_name}",
        "train_rows": len(train_bkt),
        "valid_rows": len(valid_bkt),
        "train_students": train_bkt["user_id"].nunique(),
        "valid_students": valid_bkt["user_id"].nunique(),
        "train_accuracy": safe_evaluate(model, train_bkt, "accuracy"),
        "valid_accuracy": safe_evaluate(model, valid_bkt, "accuracy"),
        "train_auc": safe_evaluate(model, train_bkt, "auc"),
        "valid_auc": safe_evaluate(model, valid_bkt, "auc"),
        "train_rmse": safe_evaluate(model, train_bkt, "rmse"),
        "valid_rmse": safe_evaluate(model, valid_bkt, "rmse"),
        **diag,
    }

    preds_valid = safe_predict(model, valid_bkt)

    return {
        "summary": summary,
        "model": model,
        "train_df": train_df,
        "valid_df": valid_df,
        "train_prepared": train_prepared,
        "valid_prepared": valid_prepared,
        "train_bkt": train_bkt,
        "valid_bkt": valid_bkt,
        "param_df": param_df,
        "param_pivot": param_pivot,
        "preds_valid": preds_valid,
    }


## 4. Build the pyBKT-ready datasets and inspect them

In [ ]:

prepared_tables = {}
bkt_tables = {}
dataset_summary_rows = []

for dataset_name, df_raw in datasets_raw.items():
    prepared_df, bkt_df = build_pybkt_df(
        prepare_columns(df_raw),
        require_equals_in_step=REQUIRE_EQUALS_IN_STEP,
    )
    prepared_tables[dataset_name] = prepared_df
    bkt_tables[dataset_name] = bkt_df

    dataset_summary_rows.append({
        "dataset_name": dataset_name,
        "prepared_rows": len(prepared_df),
        "bkt_rows": len(bkt_df),
        "students": bkt_df["user_id"].nunique(),
        "skills": bkt_df["skill_name_std"].nunique(),
    })

dataset_summary_df = pd.DataFrame(dataset_summary_rows)
display(dataset_summary_df)


## 5. Descriptive graphs

In [ ]:

# 5.1 Rows per dataset after preprocessing
plt.figure(figsize=(7, 4))
plt.bar(dataset_summary_df["dataset_name"], dataset_summary_df["bkt_rows"])
plt.title("Rows per candidate dataset")
plt.xlabel("Dataset")
plt.ylabel("BKT rows")
plt.tight_layout()
plt.show()


In [ ]:

# 5.2 First-attempt correctness rate by KC for each dataset
for dataset_name, prepared_df in prepared_tables.items():
    correct_rate = (
        prepared_df.groupby("skill_name_std")["correct"]
        .mean()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(8, 4))
    plt.bar(correct_rate.index, correct_rate.values)
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"First-attempt correct rate per KC - {dataset_name}")
    plt.xlabel("KC")
    plt.ylabel("Correct rate")
    plt.tight_layout()
    plt.show()

    display(
        prepared_df.groupby("skill_name_std")["correct"]
        .agg(n_rows="count", correct_rate="mean", n_correct="sum")
        .sort_values("correct_rate", ascending=False)
    )


In [ ]:

# 5.3 Interaction-length histograms by KC for each dataset
for dataset_name, prepared_df in prepared_tables.items():
    interaction_summary = summarize_interactions_per_student_skill(prepared_df)
    skills = sorted(interaction_summary["skill_name_std"].unique())
    n_skills = len(skills)
    n_cols = 2
    n_rows = int(np.ceil(n_skills / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, skill in zip(axes, skills):
        sub = interaction_summary[interaction_summary["skill_name_std"] == skill]
        ax.hist(sub["n_interactions"], bins=20)
        ax.set_title(f"{dataset_name} - {skill}")
        ax.set_xlabel("Interactions per student")
        ax.set_ylabel("Students")

    for ax in axes[len(skills):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## 6. Train and evaluate all dataset-model combinations

In [ ]:

all_outputs = []
all_summaries = []

for dataset_name, df_raw in datasets_raw.items():
    for model_name, fit_kwargs in MODEL_SPECS.items():
        print("=" * 100)
        print(f"Fitting dataset={dataset_name} | model={model_name}")

        out = fit_one_dataset_model(
            dataset_name=dataset_name,
            df_raw=df_raw,
            model_name=model_name,
            model_fit_kwargs=fit_kwargs,
            valid_size=VALID_SIZE,
            seed=RANDOM_STATE,
            num_fits=NUM_FITS,
        )

        all_outputs.append(out)
        all_summaries.append(out["summary"])

results_df = pd.DataFrame(all_summaries)
display(results_df)


## 7. Compare candidates and choose the best one

In [ ]:

results_ranked = results_df.copy()
results_ranked["model_complexity"] = results_ranked["model_name"].map(MODEL_COMPLEXITY)

results_ranked = results_ranked.sort_values(
    by=["valid_auc", "valid_rmse", "mean_guess", "model_complexity"],
    ascending=[False, True, True, True],
).reset_index(drop=True)

display(results_ranked)


In [ ]:

# Save the comparison table
comparison_path = OUTPUTS_DIR / "bkt_dataset_model_comparison.csv"
results_ranked.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)


In [ ]:

# 7.1 Validation metric comparison
plot_df = results_ranked[["candidate_name", "valid_accuracy", "valid_auc", "valid_rmse"]].copy()

x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(11, 5))
plt.bar(x - width, plot_df["valid_accuracy"], width=width, label="Validation accuracy")
plt.bar(x, plot_df["valid_auc"], width=width, label="Validation AUC")
plt.bar(x + width, plot_df["valid_rmse"], width=width, label="Validation RMSE")

plt.xticks(x, plot_df["candidate_name"], rotation=45, ha="right")
plt.title("Validation metrics by dataset-model candidate")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# 7.2 Parameter sanity comparison
plot_df = results_ranked[["candidate_name", "mean_guess", "max_guess", "mean_slip", "max_slip", "mean_learn"]].copy()

x = np.arange(len(plot_df))
width = 0.16

plt.figure(figsize=(12, 5))
for i, col in enumerate(["mean_guess", "max_guess", "mean_slip", "max_slip", "mean_learn"]):
    plt.bar(x + (i - 2) * width, plot_df[col], width=width, label=col)

plt.xticks(x, plot_df["candidate_name"], rotation=45, ha="right")
plt.title("Parameter diagnostics by dataset-model candidate")
plt.ylabel("Parameter value")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

output_by_candidate = {out["summary"]["candidate_name"]: out for out in all_outputs}

best_candidate_name = results_ranked.iloc[0]["candidate_name"]
print("Automatically selected best candidate:", best_candidate_name)

best_output = output_by_candidate[best_candidate_name]
print("Best dataset:", best_output["summary"]["dataset_name"])
print("Best model:", best_output["summary"]["model_name"])


## 8. Show the parameter probabilities for the best dataset

In [ ]:

best_param_df = best_output["param_df"].copy()
display(best_param_df)


In [ ]:

# Cleaner long-format table
best_param_df = best_param_df.rename(columns={
    "skill_name_std": "skill_name",
    "parameter": "parameter",
    "value": "probability",
})

display(best_param_df.sort_values(["skill_name", "parameter"]))


In [ ]:

# Wide table: one row per skill
best_params_wide = (
    best_param_df.pivot_table(
        index="skill_name",
        columns="parameter",
        values="probability",
        aggfunc="first",
    )
    .reset_index()
)

display(best_params_wide)


In [ ]:

# Save parameter tables
best_params_long_path = OUTPUTS_DIR / f"{best_candidate_name}_best_params_long.csv"
best_params_wide_path = OUTPUTS_DIR / f"{best_candidate_name}_best_params_wide.csv"

best_param_df.to_csv(best_params_long_path, index=False)
best_params_wide.to_csv(best_params_wide_path, index=False)

print("Saved:", best_params_long_path)
print("Saved:", best_params_wide_path)


In [ ]:

# Heatmap-style parameter plot for the best candidate
param_heatmap = best_params_wide.set_index("skill_name")[[c for c in ["prior", "learns", "guesses", "slips", "forgets"] if c in best_params_wide.columns]]

fig, ax = plt.subplots(figsize=(8, max(3, len(param_heatmap) * 0.8)))
im = ax.imshow(param_heatmap.values, aspect="auto")

ax.set_xticks(np.arange(param_heatmap.shape[1]))
ax.set_xticklabels(param_heatmap.columns)
ax.set_yticks(np.arange(param_heatmap.shape[0]))
ax.set_yticklabels(param_heatmap.index)
ax.set_title(f"Parameter probabilities - {best_candidate_name}")

for i in range(param_heatmap.shape[0]):
    for j in range(param_heatmap.shape[1]):
        ax.text(j, i, f"{param_heatmap.iloc[i, j]:.3f}", ha="center", va="center")

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:

# One bar plot per parameter
for parameter in ["prior", "learns", "guesses", "slips", "forgets"]:
    if parameter not in best_param_df.columns and parameter not in best_params_wide.columns:
        continue
    if parameter not in best_params_wide.columns:
        continue

    sub = best_params_wide[["skill_name", parameter]].dropna().copy()
    if len(sub) == 0:
        continue

    plt.figure(figsize=(8, 4))
    plt.bar(sub["skill_name"], sub[parameter])
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.title(f"{parameter.capitalize()} by skill - {best_candidate_name}")
    plt.xlabel("Skill")
    plt.ylabel("Probability")
    plt.tight_layout()
    plt.show()


## 9. Optional prediction-level inspection for the best dataset

In [ ]:

preds_valid = best_output["preds_valid"]

if preds_valid is None:
    print("No prediction DataFrame available for this environment/version.")
else:
    print("Prediction columns:")
    print(preds_valid.columns.tolist())
    display(preds_valid.head())

    prediction_cols = [
        c for c in preds_valid.columns
        if ("pred" in c.lower()) or ("prob" in c.lower())
    ]

    if len(prediction_cols) == 0:
        print("No probability-like columns detected automatically.")
    else:
        for col in prediction_cols:
            values = pd.to_numeric(preds_valid[col], errors="coerce").dropna()
            if len(values) == 0:
                continue
            plt.figure(figsize=(7, 4))
            plt.hist(values, bins=30)
            plt.title(f"Distribution of {col} - {best_candidate_name}")
            plt.xlabel(col)
            plt.ylabel("Rows")
            plt.tight_layout()
            plt.show()


## 10. Refit the final selected model on the full best dataset

In [ ]:

FINAL_CANDIDATE_NAME = best_candidate_name
print("Final candidate selected:", FINAL_CANDIDATE_NAME)

final_dataset_name = best_output["summary"]["dataset_name"]
final_model_name = best_output["summary"]["model_name"]
final_fit_kwargs = MODEL_SPECS[final_model_name]

final_prepared, final_bkt = build_pybkt_df(
    prepare_columns(datasets_raw[final_dataset_name]),
    require_equals_in_step=REQUIRE_EQUALS_IN_STEP,
)

final_model = Model(seed=RANDOM_STATE, num_fits=NUM_FITS)
final_model.fit(data=final_bkt, defaults=DEFAULTS, **final_fit_kwargs)

final_params = extract_param_table(final_model).rename(columns={
    "skill_name_std": "skill_name",
    "value": "probability",
})

display(final_params)


In [ ]:

# Save outputs
final_params_path = OUTPUTS_DIR / f"{FINAL_CANDIDATE_NAME}_final_params.csv"
final_bkt_path = OUTPUTS_DIR / f"{FINAL_CANDIDATE_NAME}_bkt_input.csv"
final_model_path = MODELS_DIR / f"{FINAL_CANDIDATE_NAME}_pybkt_model.pkl"

final_params.to_csv(final_params_path, index=False)
final_bkt.to_csv(final_bkt_path, index=False)

with open(final_model_path, "wb") as f:
    pickle.dump(final_model, f)

print("Saved:", final_params_path)
print("Saved:", final_bkt_path)
print("Saved:", final_model_path)


## 11. Why the selected model

The forgetting variant improved validation AUC by ~4 points consistently across all datasets with no accuracy penalty — makes sense for algebra where students genuinely regress between sessions. Dataset choice (2006-07 vs 2008-09 vs merged) matters less than the model variant; the final selection is driven by which combination gives the lowest cross-validated RMSE with the most students.